# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape); print(df.dtypes); df.head(3)

(30000, 44)
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
ag

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


In [3]:
import numpy as np, os, json
df = df[df.impressions_90d > 0].copy()

def bucket_table(d, signal, outcome, bins):
    d = d[[signal, outcome]].dropna().copy()
    d["bucket"] = pd.cut(d[signal], bins)
    t = d.groupby("bucket", observed=True)[outcome].agg(n="size", mean="mean", median="median")
    print(f"\n{signal} -> {outcome}"); print(t); return t

tA = bucket_table(df, "days_since_last_update", "trend_pct", [-1, 90, 180, 365, 730, 99999])
tB = bucket_table(df, "avg_position", "ctr", [0, 3, 10, 20, 50, 1000])


days_since_last_update -> trend_pct
                n       mean  median
bucket                              
(-1, 90]    17621   0.707094   -33.3
(90, 180]    8858 -15.683224   -34.4
(180, 365]    130  -4.718462   -44.2
(365, 730]      3 -96.166667  -100.0

avg_position -> ctr
                n      mean  median
bucket                             
(0, 3]       1141  2.714303    0.00
(3, 10]     11842  0.651045    0.16
(10, 20]     7273  0.323443    0.10
(20, 50]     7225  0.222345    0.03
(50, 1000]   1314  0.150784    0.00


### Signal checks

**Signal A: staleness (days_since_last_update) -> trend_pct. Verdict: MIXED**

Observed: median trend is negative in every bucket and gets worse for older pages (-33.3, -34.4, -44.2, -100). But the first two buckets are nearly identical, the means are not monotonic (+0.7, -15.7, -4.7, -96.2), and the oldest buckets are tiny (n=130 and n=3). So the direction is only directional. Staleness stays in the rule with a small weight.

**Signal B: CTR vs position (avg_position -> ctr). Verdict: CONFIRMED**

Observed: mean CTR falls steadily as position gets worse: 2.71 (pos 1-3), 0.65 (4-10), 0.32 (11-20), 0.22 (21-50), 0.15 (50+). Every bucket has a large n (1,141 to 11,842). This is the base of the CTR-fix logic (flag-linked).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
d = df.copy()
d["ctr_gap"] = (d.groupby("position_tier")["ctr"].transform("median") - d["ctr"]).clip(lower=0)
d["p_stale"] = d["days_since_last_update"].rank(pct=True)
d["p_vol"]   = d["impressions_90d"].rank(pct=True)
d["p_gap"]   = d["ctr_gap"].rank(pct=True)

w = {"stale": 0.15, "vol": 0.45, "gap": 0.40}
d["score"] = w["stale"]*d.p_stale + w["vol"]*d.p_vol + w["gap"]*d.p_gap

comp = np.column_stack([w["stale"]*d.p_stale, w["vol"]*d.p_vol, w["gap"]*d.p_gap]).argmax(axis=1)
d["reason_code"] = np.array(["STALE_PAGE", "HIGH_VOLUME", "CTR_BELOW_PEER"])[comp]
d["action"] = d["reason_code"].map({"STALE_PAGE": "REFRESH",
                                    "HIGH_VOLUME": "QUICK_WIN_OPTIMIZE",
                                    "CTR_BELOW_PEER": "FIX_TITLE_META"})

queue = d.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
os.makedirs("work/outputs", exist_ok=True)
queue[["rank", "content_id", "score", "reason_code", "action"]].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue.reason_code.value_counts())

reason_code
HIGH_VOLUME       16715
CTR_BELOW_PEER    13040
STALE_PAGE          245
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
wrong = {
    "STALE_PAGE": "traffic is seasonal or the topic is evergreen, so a refresh changes nothing",
"HIGH_VOLUME": "intent mismatch (impressions do not mean the page answers the query), so extra clicks may not convert",
    "CTR_BELOW_PEER": "low CTR is caused by SERP features or ads, so a new title will not help",
}
top = queue.head(20).copy()
n_high = (top[["p_stale", "p_vol", "p_gap"]] > 0.8).sum(axis=1)
top["confidence"] = np.where(n_high >= 2, "high", np.where(n_high == 1, "medium", "low"))
top["peer_ctr"] = (top.ctr + top.ctr_gap).round(2)

def risk(r):
    extra = []
    if r.ctr == 0:
        extra.append("zero CTR may be an impression artifact, not a real missed click")
    if r.avg_position >= 9:
        extra.append("position near the page-1 edge may drop off, so gains are fragile")
    return "; ".join([wrong[r.reason_code]] + extra)

top["wrong_if"] = top.apply(risk, axis=1)
top["why"] = top.apply(lambda r: f"{int(r.impressions_90d):,} impressions in 90d at position {r.avg_position}, "
                                 f"CTR {r.ctr} vs peer median {r.peer_ctr}", axis=1)

for _, r in top.iterrows():
    print(f"#{r['rank']} {r.content_id} | {r.action} | {r.why} | confidence: {r.confidence} | wrong if: {r.wrong_if}\n")


#1 content_c8e9d6ab9013 | QUICK_WIN_OPTIMIZE | 208,678 impressions in 90d at position 9.7, CTR 0.0 vs peer median 0.16 | confidence: high | wrong if: intent mismatch (impressions do not mean the page answers the query), so extra clicks may not convert; zero CTR may be an impression artifact, not a real missed click; position near the page-1 edge may drop off, so gains are fragile

#2 content_c1fe78bc4e37 | QUICK_WIN_OPTIMIZE | 134,055 impressions in 90d at position 7.5, CTR 0.03 vs peer median 0.16 | confidence: high | wrong if: intent mismatch (impressions do not mean the page answers the query), so extra clicks may not convert

#3 content_b115f7c74779 | QUICK_WIN_OPTIMIZE | 123,469 impressions in 90d at position 8.0, CTR 0.03 vs peer median 0.16 | confidence: high | wrong if: intent mismatch (impressions do not mean the page answers the query), so extra clicks may not convert

#4 content_d0cc5baa4995 | QUICK_WIN_OPTIMIZE | 83,651 impressions in 90d at position 6.6, CTR 0.03 vs peer m

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: rows with CTR of 0.00 despite very high impressions (for example rank 1 with 208k impressions at position 9.7) look suspicious. They may be impression artifacts rather than real missed clicks. Also, the top-20 has only one reason code, so the rule is dominated by volume, and staleness (weight 0.15) adds almost nothing at the top. Fix for Week 5: cap or log-scale impressions so one signal does not dominate.

Leakage check: rule inputs are days_since_last_update, impressions_90d, ctr and position_tier. No future-window columns (last_30d, prev_30d) and no label-derived columns. trend_pct was used only as the outcome in the Signal A check, not in the rule.

In [11]:
used = ["days_since_last_update", "impressions_90d", "ctr", "position_tier"]
forbidden = ["trend_pct", "trend_direction", "clicks_last_30d", "sessions_last_30d",
             "impressions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
print("Rule inputs:", used)
print("Leak columns used:", [c for c in used if c in forbidden])  # must be []

print(top.reason_code.value_counts())
print("Staleness range in top-20:", top.days_since_last_update.min(), "to", top.days_since_last_update.max())
weak = top[(top.ctr <= 0.01) | (top.avg_position >= 9)]
print(len(weak), "weak-looking picks")
weak[["rank", "content_id", "action", "avg_position", "impressions_90d", "ctr"]]

json.dump({"n_rows": len(queue), "reason_counts": queue.reason_code.value_counts().to_dict(),
           "signalA_verdict": "MIXED", "signalB_verdict": "CONFIRMED"},
          open("work/outputs/w04_metrics.json", "w"), indent=2)


Rule inputs: ['days_since_last_update', 'impressions_90d', 'ctr', 'position_tier']
Leak columns used: []
reason_code
HIGH_VOLUME    20
Name: count, dtype: int64
Staleness range in top-20: 104 to 104
5 weak-looking picks


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.